In [2]:
# ============================================================
# CELL 1 - INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install -q xgboost scikit-learn pandas numpy matplotlib seaborn joblib

In [3]:
# ============================================================
# CELL 2 - IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor

import joblib
import warnings

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [4]:
# ============================================================
# CELL 3 - UPLOAD DATASET
# ============================================================

from google.colab import files

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

print("Uploaded file:", file_name)

df = pd.read_csv(file_name)

print("\nDataset loaded successfully!")
print("Shape:", df.shape)

Saving Merged_Stock_Market_Dataset.csv to Merged_Stock_Market_Dataset.csv
Uploaded file: Merged_Stock_Market_Dataset.csv

Dataset loaded successfully!
Shape: (177728, 10)


In [5]:
# ============================================================
# CELL 4 - INSPECT DATASET
# ============================================================

print("First 5 rows:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

First 5 rows:


,Date,Ticker,Company,Open,High,Low,Close,Adj Close,Volume,Source
0,2016-09-20,6758.T,6758.T,655.400024,674.799988,654.799988,664.000000,623.805298,32968000.0,Yahoo Finance Worldwide
1,2016-09-21,6758.T,6758.T,663.000000,674.799988,655.000000,674.599976,633.763489,35469500.0,Yahoo Finance Worldwide
2,2016-09-23,6758.T,6758.T,680.000000,685.799988,678.000000,680.000000,638.836609,31894000.0,Yahoo Finance Worldwide
3,2016-09-26,6758.T,6758.T,679.599976,682.000000,673.400024,676.200012,635.266724,20983500.0,Yahoo Finance Worldwide
4,2016-09-27,6758.T,6758.T,660.000000,676.000000,660.000000,675.599976,634.702942,30895500.0,Yahoo Finance Worldwide



Column names:
['Date', 'Ticker', 'Company', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Source']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 177728 entries, 0 to 177727
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   Date       177728 non-null  object 
 1   Ticker     177728 non-null  object 
 2   Company    177728 non-null  object 
 3   Open       177728 non-null  float64
 4   High       177728 non-null  float64
 5   Low        177728 non-null  float64
 6   Close      177728 non-null  float64
 7   Adj Close  177728 non-null  float64
 8   Volume     177728 non-null  float64
 9   Source     177728 non-null  object 
dtypes: float64(6), object(4)
memory usage: 13.6+ MB

Missing values:


,0
Date,0
Ticker,0
Company,0
Open,0
High,0
Low,0
Close,0
Adj Close,0
Volume,0
Source,0



Duplicate rows: 0


In [6]:
# ============================================================
# CELL 5 - CLEAN COLUMN NAMES
# ============================================================

df.columns = df.columns.str.strip()

print("Cleaned columns:")
print(df.columns.tolist())

Cleaned columns:
['Date', 'Ticker', 'Company', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Source']


In [7]:
# ============================================================
# CELL 6 - DATE PREPROCESSING AND SORTING
# ============================================================

# Convert Date column to datetime
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Remove rows where Date is missing
df = df.dropna(subset=["Date"])

# Sort by Ticker and Date
df = df.sort_values(
    by=["Ticker", "Date"]
).reset_index(drop=True)

print("Date conversion completed.")

print("\nDate range:")
print("Start:", df["Date"].min())
print("End  :", df["Date"].max())

print("\nDataset shape after date cleaning:")
print(df.shape)

Date conversion completed.

Date range:
Start: 2002-08-12 00:00:00
End  : 2026-09-17 00:00:00

Dataset shape after date cleaning:
(177728, 10)


In [8]:
# ============================================================
# CELL 7 - FEATURES FOR BOTH LASSO AND XGBOOST
# USER INPUT FEATURES ONLY
# ============================================================

features = [
    "Open",
    "High",
    "Low",
    "Close",
    "Adj Close",
    "Volume"
]

print("Number of features:", len(features))

print("\nFeatures:")
for i, feature in enumerate(features, 1):
    print(i, "-", feature)

Number of features: 6

Features:
1 - Open
2 - High
3 - Low
4 - Close
5 - Adj Close
6 - Volume


In [9]:
# ============================================================
# CELL 8 - CONVERT NUMERIC COLUMNS
# ============================================================

numeric_columns = [
    "Open",
    "High",
    "Low",
    "Close",
    "Adj Close",
    "Volume"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Numeric conversion completed.")

display(df[numeric_columns].head())

Numeric conversion completed.


,Open,High,Low,Close,Adj Close,Volume
0,655.400024,674.799988,654.799988,664.000000,623.805298,32968000.0
1,663.000000,674.799988,655.000000,674.599976,633.763489,35469500.0
2,680.000000,685.799988,678.000000,680.000000,638.836609,31894000.0
3,679.599976,682.000000,673.400024,676.200012,635.266724,20983500.0
4,660.000000,676.000000,660.000000,675.599976,634.702942,30895500.0


In [10]:
# ============================================================
# CELL 9 - HANDLE MISSING VALUES
# ============================================================

print("Missing values before cleaning:")
display(df[features].isnull().sum())

# Remove rows where any model feature is missing
df = df.dropna(subset=features)

print("\nMissing values after cleaning:")
display(df[features].isnull().sum())

print("\nDataset shape:")
print(df.shape)

Missing values before cleaning:


,0
Open,0
High,0
Low,0
Close,0
Adj Close,0
Volume,0



Missing values after cleaning:


,0
Open,0
High,0
Low,0
Close,0
Adj Close,0
Volume,0



Dataset shape:
(177728, 10)


In [11]:
# ============================================================
# CELL 10 - CREATE NEXT-DAY CLOSE TARGET
# ============================================================

# For each ticker:
# Target = next trading day's Close

df["Target_Next_Close"] = (
    df.groupby("Ticker")["Close"].shift(-1)
)

print("Next-day target created.")

display(
    df[
        [
            "Date",
            "Ticker",
            "Company",
            "Close",
            "Target_Next_Close"
        ]
    ].head(10)
)

Next-day target created.


,Date,Ticker,Company,Close,Target_Next_Close
0,2016-09-20,6758.T,6758.T,664.000000,674.599976
1,2016-09-21,6758.T,6758.T,674.599976,680.000000
2,2016-09-23,6758.T,6758.T,680.000000,676.200012
3,2016-09-26,6758.T,6758.T,676.200012,675.599976
4,2016-09-27,6758.T,6758.T,675.599976,673.599976
5,2016-09-28,6758.T,6758.T,673.599976,676.599976
6,2016-09-29,6758.T,6758.T,676.599976,658.599976
7,2016-09-30,6758.T,6758.T,658.599976,667.000000
8,2016-10-03,6758.T,6758.T,667.000000,674.200012
9,2016-10-04,6758.T,6758.T,674.200012,678.400024


In [12]:
# ============================================================
# CELL 11 - REMOVE FINAL ROWS WITHOUT TARGET
# ============================================================

print("Rows before removing missing target:", len(df))

df_model = df.dropna(
    subset=["Target_Next_Close"]
).copy()

print("Rows after removing missing target:", len(df_model))

print("\nFinal modeling dataset shape:")
print(df_model.shape)

Rows before removing missing target: 177728
Rows after removing missing target: 177658

Final modeling dataset shape:
(177658, 11)


In [13]:
# ============================================================
# CELL 12 - PREPARE FEATURES AND TARGET
# ============================================================

X = df_model[features].copy()

y = df_model["Target_Next_Close"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nX columns:")
print(X.columns.tolist())

print("\nTarget:")
print("Target_Next_Close")

X shape: (177658, 6)
y shape: (177658,)

X columns:
['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']

Target:
Target_Next_Close


In [14]:
# ============================================================
# CELL 13 - CHRONOLOGICAL TRAIN / TEST SPLIT
# ============================================================

# Get unique dates
unique_dates = np.sort(
    df_model["Date"].unique()
)

# 80% date point
split_index = int(len(unique_dates) * 0.80)

split_date = unique_dates[split_index]

print("Split date:", split_date)

# Training data
train_mask = df_model["Date"] < split_date

# Testing data
test_mask = df_model["Date"] >= split_date

X_train = df_model.loc[train_mask, features]
y_train = df_model.loc[train_mask, "Target_Next_Close"]

X_test = df_model.loc[test_mask, features]
y_test = df_model.loc[test_mask, "Target_Next_Close"]

print("\nTraining samples:", len(X_train))
print("Testing samples :", len(X_test))

print("\nTraining date range:")
print(
    df_model.loc[train_mask, "Date"].min(),
    "to",
    df_model.loc[train_mask, "Date"].max()
)

print("\nTesting date range:")
print(
    df_model.loc[test_mask, "Date"].min(),
    "to",
    df_model.loc[test_mask, "Date"].max()
)

Split date: 2022-01-07T00:00:00.000000000

Training samples: 95786
Testing samples : 81872

Training date range:
2002-08-12 00:00:00 to 2022-01-06 00:00:00

Testing date range:
2022-01-07 00:00:00 to 2026-09-16 00:00:00


In [15]:
# ============================================================
# CELL 14 - FEATURE SCALING FOR LASSO
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print("Scaling completed.")

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled testing shape :", X_test_scaled.shape)

Scaling completed.
Scaled training shape: (95786, 6)
Scaled testing shape : (81872, 6)


In [16]:
# ============================================================
# CELL 15 - TRAIN LASSO REGRESSION
# ============================================================

lasso_model = Lasso(
    alpha=0.001,
    max_iter=10000,
    random_state=42
)

lasso_model.fit(
    X_train_scaled,
    y_train
)

print("Lasso model trained successfully.")

Lasso model trained successfully.


In [17]:
# ============================================================
# CELL 16 - LASSO PREDICTION
# ============================================================

lasso_pred = lasso_model.predict(
    X_test_scaled
)

print("Lasso predictions generated.")

print("\nFirst 10 Lasso predictions:")
print(lasso_pred[:10])

Lasso predictions generated.

First 10 Lasso predictions:
[2908.79846659 2885.57734446 2913.55864487 2898.77177588 2840.24957456
 2846.90655071 2852.24039006 2529.86152456 2611.89981703 2573.18869861]


In [18]:
# ============================================================
# CELL 17 - TRAIN XGBOOST REGRESSION
# ============================================================

xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train,
    y_train
)

print("XGBoost model trained successfully.")

XGBoost model trained successfully.


In [19]:
# ============================================================
# CELL 18 - XGBOOST PREDICTION
# ============================================================

xgb_pred = xgb_model.predict(
    X_test
)

print("XGBoost predictions generated.")

print("\nFirst 10 XGBoost predictions:")
print(xgb_pred[:10])

XGBoost predictions generated.

First 10 XGBoost predictions:
[2849.3218 2849.3218 2857.0515 2859.5334 2849.3218 2859.5334 2854.9773
 2485.0269 2552.2947 2552.2947]


In [20]:
# ============================================================
# CELL 19 - ENSEMBLE PREDICTION
# LASSO = 60%
# XGBOOST = 40%
# ============================================================

LASSO_WEIGHT = 0.60
XGB_WEIGHT = 0.40

ensemble_pred = (
    LASSO_WEIGHT * lasso_pred
    +
    XGB_WEIGHT * xgb_pred
)

print("Ensemble prediction generated.")

print("\nFormula:")
print("Ensemble = 0.60 × Lasso + 0.40 × XGBoost")

print("\nFirst 10 ensemble predictions:")
print(ensemble_pred[:10])

Ensemble prediction generated.

Formula:
Ensemble = 0.60 × Lasso + 0.40 × XGBoost

First 10 ensemble predictions:
[2885.00783972 2871.07516644 2890.95586563 2883.07642002 2843.8785045
 2851.95728492 2853.33520083 2511.92765692 2588.05779793 2564.83112688]


In [21]:
# ============================================================
# CELL 20 - EVALUATION FUNCTION
# ============================================================

def calculate_metrics(actual, predicted):

    mae = mean_absolute_error(
        actual,
        predicted
    )

    mse = mean_squared_error(
        actual,
        predicted
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        actual,
        predicted
    )

    # Avoid division by zero
    non_zero_mask = actual != 0

    mape = np.mean(
        np.abs(
            (
                actual[non_zero_mask]
                -
                predicted[non_zero_mask]
            )
            /
            actual[non_zero_mask]
        )
    ) * 100

    return mae, mse, rmse, r2, mape

In [22]:
# ============================================================
# CELL 21 - MODEL EVALUATION
# ============================================================

lasso_metrics = calculate_metrics(
    y_test.values,
    lasso_pred
)

xgb_metrics = calculate_metrics(
    y_test.values,
    xgb_pred
)

ensemble_metrics = calculate_metrics(
    y_test.values,
    ensemble_pred
)

results_metrics = pd.DataFrame({
    "Model": [
        "Lasso",
        "XGBoost",
        "Ensemble"
    ],
    "MAE": [
        lasso_metrics[0],
        xgb_metrics[0],
        ensemble_metrics[0]
    ],
    "MSE": [
        lasso_metrics[1],
        xgb_metrics[1],
        ensemble_metrics[1]
    ],
    "RMSE": [
        lasso_metrics[2],
        xgb_metrics[2],
        ensemble_metrics[2]
    ],
    "R2": [
        lasso_metrics[3],
        xgb_metrics[3],
        ensemble_metrics[3]
    ],
    "MAPE (%)": [
        lasso_metrics[4],
        xgb_metrics[4],
        ensemble_metrics[4]
    ]
})

display(
    results_metrics.round(4)
)

,Model,MAE,MSE,RMSE,R2,MAPE (%)
0,Lasso,29.3893,2.866213e+04,169.2989,0.9995,1.4519
1,XGBoost,252.9612,2.257999e+06,1502.6640,0.9602,2.9381
2,Ensemble,109.0081,3.779666e+05,614.7899,0.9933,1.9560


In [23]:
# ============================================================
# CELL 22 - ACTUAL VS PREDICTED RESULTS
# ============================================================

test_results = df_model.loc[
    test_mask,
    [
        "Date",
        "Ticker",
        "Company",
        "Open",
        "High",
        "Low",
        "Close",
        "Adj Close",
        "Volume",
        "Target_Next_Close"
    ]
].copy()

test_results = test_results.reset_index(
    drop=True
)

test_results["Lasso_Predicted_Next_Close"] = lasso_pred

test_results["XGBoost_Predicted_Next_Close"] = xgb_pred

test_results["Ensemble_Predicted_Next_Close"] = ensemble_pred

test_results.rename(
    columns={
        "Target_Next_Close": "Actual_Next_Close"
    },
    inplace=True
)

print("Actual vs predicted results:")
display(
    test_results.head(20)
)

Actual vs predicted results:


,Date,Ticker,Company,Open,High,Low,Close,Adj Close,Volume,Actual_Next_Close,Lasso_Predicted_Next_Close,XGBoost_Predicted_Next_Close,Ensemble_Predicted_Next_Close
0,2022-01-07,6758.T,6758.T,2900.0,2946.0,2848.0,2908.0,2823.538818,34978000.0,2881.0,2908.798467,2849.321777,2885.007840
1,2022-01-11,6758.T,6758.T,2854.0,2919.0,2848.0,2881.0,2797.323242,24169500.0,2904.0,2885.577344,2849.321777,2871.075166
2,2022-01-12,6758.T,6758.T,2927.0,2940.0,2904.0,2904.0,2819.655029,19037000.0,2894.0,2913.558645,2857.051514,2890.955866
3,2022-01-13,6758.T,6758.T,2896.0,2922.0,2883.0,2894.0,2809.945312,12115500.0,2838.0,2898.771776,2859.533447,2883.076420
4,2022-01-14,6758.T,6758.T,2866.0,2869.0,2800.0,2838.0,2755.572021,21353000.0,2847.0,2840.249575,2849.321777,2843.878505
5,2022-01-17,6758.T,6758.T,2843.0,2866.0,2825.0,2847.0,2764.310547,10714000.0,2846.0,2846.906551,2859.533447,2851.957285
6,2022-01-18,6758.T,6758.T,2870.0,2883.0,2821.0,2846.0,2763.339844,13572000.0,2482.0,2852.240390,2854.977295,2853.335201
7,2022-01-19,6758.T,6758.T,2620.0,2621.0,2480.0,2482.0,2409.911377,74886000.0,2627.0,2529.861525,2485.026855,2511.927657
8,2022-01-20,6758.T,6758.T,2602.0,2627.0,2550.0,2627.0,2550.700195,48368500.0,2591.0,2611.899817,2552.294678,2588.057798
9,2022-01-21,6758.T,6758.T,2540.0,2592.0,2496.0,2591.0,2515.745850,37811500.0,2567.0,2573.188699,2552.294678,2564.831127


In [24]:
# ============================================================
# CELL 23 - CALCULATE ENSEMBLE ERROR
# ============================================================

test_results["Ensemble_Error"] = (
    test_results["Actual_Next_Close"]
    -
    test_results["Ensemble_Predicted_Next_Close"]
)

test_results["Absolute_Error"] = np.abs(
    test_results["Ensemble_Error"]
)

display(
    test_results.head(20)
)

,Date,Ticker,Company,Open,High,Low,Close,Adj Close,Volume,Actual_Next_Close,Lasso_Predicted_Next_Close,XGBoost_Predicted_Next_Close,Ensemble_Predicted_Next_Close,Ensemble_Error,Absolute_Error
0,2022-01-07,6758.T,6758.T,2900.0,2946.0,2848.0,2908.0,2823.538818,34978000.0,2881.0,2908.798467,2849.321777,2885.007840,-4.007840,4.007840
1,2022-01-11,6758.T,6758.T,2854.0,2919.0,2848.0,2881.0,2797.323242,24169500.0,2904.0,2885.577344,2849.321777,2871.075166,32.924834,32.924834
2,2022-01-12,6758.T,6758.T,2927.0,2940.0,2904.0,2904.0,2819.655029,19037000.0,2894.0,2913.558645,2857.051514,2890.955866,3.044134,3.044134
3,2022-01-13,6758.T,6758.T,2896.0,2922.0,2883.0,2894.0,2809.945312,12115500.0,2838.0,2898.771776,2859.533447,2883.076420,-45.076420,45.076420
4,2022-01-14,6758.T,6758.T,2866.0,2869.0,2800.0,2838.0,2755.572021,21353000.0,2847.0,2840.249575,2849.321777,2843.878505,3.121495,3.121495
5,2022-01-17,6758.T,6758.T,2843.0,2866.0,2825.0,2847.0,2764.310547,10714000.0,2846.0,2846.906551,2859.533447,2851.957285,-5.957285,5.957285
6,2022-01-18,6758.T,6758.T,2870.0,2883.0,2821.0,2846.0,2763.339844,13572000.0,2482.0,2852.240390,2854.977295,2853.335201,-371.335201,371.335201
7,2022-01-19,6758.T,6758.T,2620.0,2621.0,2480.0,2482.0,2409.911377,74886000.0,2627.0,2529.861525,2485.026855,2511.927657,115.072343,115.072343
8,2022-01-20,6758.T,6758.T,2602.0,2627.0,2550.0,2627.0,2550.700195,48368500.0,2591.0,2611.899817,2552.294678,2588.057798,2.942202,2.942202
9,2022-01-21,6758.T,6758.T,2540.0,2592.0,2496.0,2591.0,2515.745850,37811500.0,2567.0,2573.188699,2552.294678,2564.831127,2.168873,2.168873


In [25]:
# ============================================================
# CELL 26 - XGBOOST FEATURE IMPORTANCE
# ============================================================

feature_importance = pd.DataFrame({
    "Feature": features,
    "Importance": xgb_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

display(feature_importance)

,Feature,Importance
2,Low,0.567631
1,High,0.283730
3,Close,0.055143
4,Adj Close,0.046666
0,Open,0.044809
5,Volume,0.002021


In [26]:
# ============================================================
# CELL 28 - DIRECTIONAL ACCURACY
# ============================================================

actual_direction = (
    test_results["Actual_Next_Close"]
    >
    test_results["Close"]
)

predicted_direction = (
    test_results["Ensemble_Predicted_Next_Close"]
    >
    test_results["Close"]
)

directional_accuracy = (
    actual_direction == predicted_direction
).mean() * 100

print(
    f"Ensemble Directional Accuracy: "
    f"{directional_accuracy:.2f}%"
)

Ensemble Directional Accuracy: 50.06%


In [29]:
# ============================================================
# CELL 32 - FINAL USER INPUT PREDICTION
# ============================================================

print("=" * 65)
print("       NEXT TRADING DAY CLOSE PRICE PREDICTION")
print("=" * 65)

# ------------------------------------------------------------
# USER INPUT
# ------------------------------------------------------------

ticker_input = input("Enter Ticker: ")


open_input = float(
    input("Enter Open Price: ")
)

high_input = float(
    input("Enter High Price: ")
)

low_input = float(
    input("Enter Low Price: ")
)

close_input = float(
    input("Enter Close Price: ")
)

adj_close_input = float(
    input("Enter Adj Close Price: ")
)

volume_input = float(
    input("Enter Volume: ")
)

# ------------------------------------------------------------
# CREATE USER INPUT DATAFRAME
# ------------------------------------------------------------

user_input = pd.DataFrame({
    "Open": [open_input],
    "High": [high_input],
    "Low": [low_input],
    "Close": [close_input],
    "Adj Close": [adj_close_input],
    "Volume": [volume_input]
})

# Make sure columns are in exactly the training order
user_input = user_input[features]

# ------------------------------------------------------------
# LASSO PREDICTION
# ------------------------------------------------------------

user_input_scaled = scaler.transform(
    user_input
)

lasso_user_prediction = lasso_model.predict(
    user_input_scaled
)[0]

# ------------------------------------------------------------
# XGBOOST PREDICTION
# ------------------------------------------------------------

xgb_user_prediction = xgb_model.predict(
    user_input
)[0]

# ------------------------------------------------------------
# 60% LASSO + 40% XGBOOST
# ------------------------------------------------------------

ensemble_user_prediction = (
    0.60 * lasso_user_prediction
    +
    0.40 * xgb_user_prediction
)

# ------------------------------------------------------------
# DISPLAY RESULT
# ------------------------------------------------------------

print("\n")
print("=" * 65)
print("                    PREDICTION RESULT")
print("=" * 65)

print(f"Ticker              : {ticker_input}")

print("-" * 65)

print(f"Open                : {open_input:.4f}")
print(f"High                : {high_input:.4f}")
print(f"Low                 : {low_input:.4f}")
print(f"Close               : {close_input:.4f}")
print(f"Adj Close           : {adj_close_input:.4f}")
print(f"Volume              : {volume_input:.0f}")

print("-" * 65)

print(
    f"Lasso Next Close    : "
    f"{lasso_user_prediction:.4f}"
)

print(
    f"XGBoost Next Close  : "
    f"{xgb_user_prediction:.4f}"
)

print(
    f"Ensemble Next Close : "
    f"{ensemble_user_prediction:.4f}"
)

print("-" * 65)

print(
    "Ensemble Formula    : "
    "60% Lasso + 40% XGBoost"
)

print("=" * 65)

       NEXT TRADING DAY CLOSE PRICE PREDICTION
Enter Ticker: AAPL
Enter Open Price: 332.53
Enter High Price: 335.48
Enter Low Price: 330.70
Enter Close Price: 332.41
Enter Adj Close Price: 332.41
Enter Volume: 35981000


                    PREDICTION RESULT
Ticker              : AAPL
-----------------------------------------------------------------
Open                : 332.5300
High                : 335.4800
Low                 : 330.7000
Close               : 332.4100
Adj Close           : 332.4100
Volume              : 35981000
-----------------------------------------------------------------
Lasso Next Close    : 332.6272
XGBoost Next Close  : 330.5343
Ensemble Next Close : 331.7901
-----------------------------------------------------------------
Ensemble Formula    : 60% Lasso + 40% XGBoost
